# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

In [39]:
%%sh
echo "# ml-ops-hw07-git" >> README.md
git init
git add README.md
git commit -m "first commit"
git branch -M main
git remote add origin https://github.com/Lilya-te/ml-ops-hw07-git.git
git push -u origin main
pip install scikit-learn numpy pandas -qqq
pip freeze > requirements.txt

Reinitialized existing Git repository in /Users/l.tereshchenkova/MIPT/semester-2/courses/ml-ops-hw07-git/ml-ops-hw07-git/.git/
[main f1ff842] first commit
 1 file changed, 1 insertion(+)
branch 'main' set up to track 'origin/main'.


error: remote origin already exists.
To https://github.com/Lilya-te/ml-ops-hw07-git.git
   e14bc36..f1ff842  main -> main


In [52]:
!mkdir -p .github/workflows

In [41]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Overwriting ml_pipeline.py


### Проверяем работоспособность пайплайна

In [42]:
!python ml_pipeline.py

Точность аccuracy: 1.00


In [55]:
%%writefile .github/workflows/gitlab-ci.yml
name: Gitea Actions Demo
run-name: testing out Gitea Actions 🚀
on: [push]

jobs:
  Explore-Gitea-Actions:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - name: Install dependencies
        run: pip install scikit-learn numpy pandas
      - name: Run pipeline
        run: python ml_pipeline.py


Overwriting .github/workflows/gitlab-ci.yml


In [44]:
# !git add .gitlab-ci.yml ml_pipeline.py
# !git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitLab"
# !git log

### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

[gitlab-ci](https://github.com/Lilya-te/ml-ops-hw07-git/actions/workflows/gitlab-ci.yml)

## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.


*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

### Выбор стратегии деплоя для ML-сервиса

**Решение** стратегия **Blue-Green Deployment**.

**Причины**
- Нужно снизить риск простоя сервиса во время релиза: переключение трафика между `blue` и `green` выполняется почти мгновенно.
- Нужен быстрый и безопасный откат: при проблеме можно вернуть трафик на предыдущую среду.
- Для ML-сервиса важно отдельно валидировать новую версию модели перед полным переключением пользователей.
- Упрощается аудит изменений: в ADR фиксируются причины выбора стратегии и критерии успеха релиза.

**Риски, к которым приводит это решение**
- Рост инфраструктурных затрат: одновременно поддерживаются две полноценных среды.
- Риск рассинхронизации окружений (`blue` и `green`) по конфигам, секретам и версиям зависимостей.
- Более сложная операционная поддержка: требуется строгая дисциплина в мониторинге, health-check и переключении трафика.
- Возможные проблемы с состоянием (stateful-компоненты, миграции БД): не все изменения безопасно переключаются «одним рубильником».
- Ошибки маршрутизации или балансировки могут привести к частичному попаданию пользователей в некорректную среду.

**Последствия и меры снижения рисков**
- Автоматизировать проверки перед переключением трафика (smoke/integration tests).
- Применять обратимые миграции БД
- Держать единый шаблон конфигурации для обеих сред и регулярные drift-проверки.

## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

In [45]:
%%writefile docker-compose-blue.yaml
version: "3.9"

services:
  ml-blue:
    image: python:3.11-slim
    working_dir: /app
    volumes:
      - ./:/app
    command: python -m http.server 8000
    expose:
      - "8000"
    healthcheck:
      test: ["CMD-SHELL", "python -c \"import urllib.request; urllib.request.urlopen('http://localhost:8000', timeout=2)\""]
      interval: 10s
      timeout: 3s
      retries: 5

  ml-green:
    image: python:3.11-slim
    working_dir: /app
    volumes:
      - ./:/app
    command: python -m http.server 8000
    expose:
      - "8000"
    healthcheck:
      test: ["CMD-SHELL", "python -c \"import urllib.request; urllib.request.urlopen('http://localhost:8000', timeout=2)\""]
      interval: 10s
      timeout: 3s
      retries: 5

  router:
    image: nginx:alpine
    depends_on:
      ml-blue:
        condition: service_healthy
      ml-green:
        condition: service_healthy
    environment:
      ACTIVE_COLOR: ${ACTIVE_COLOR:-blue}
    ports:
      - "8000:80"
    command: >
      /bin/sh -c "
      cat > /etc/nginx/conf.d/default.conf <<EOF
      server {
        listen 80;
        location / {
          proxy_pass http://ml-$${ACTIVE_COLOR}:8000;
          proxy_set_header Host $$host;
          proxy_set_header X-Real-IP $$remote_addr;
          proxy_set_header X-Forwarded-For $$proxy_add_x_forwarded_for;
        }
      }
      EOF
      && nginx -g 'daemon off;'
      "


Overwriting docker-compose-blue.yaml


## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

In [46]:
# План A/B-тестирования для ML-модели
from math import ceil, sqrt

# 1) Гипотеза эксперимента
hypothesis = (
    "Новая модель (B) повысит конверсию в целевое действие "
    "по сравнению с текущей моделью (A)."
)

# 2) Основные параметры
alpha = 0.05          # уровень значимости
power = 0.80          # статистическая мощность
z_alpha_2 = 1.96      # квантиль для alpha/2 при норм. приближении
z_beta = 0.84         # квантиль для мощности 0.8

baseline_conversion = 0.10   # ожидаемая конверсия в контроле (A)
mde_abs = 0.02               # минимально значимый абсолютный эффект (+2 п.п.)
variant_conversion = baseline_conversion + mde_abs

traffic_split = {"A": 0.5, "B": 0.5}
daily_users = 1000

# 3) Расчет размера выборки (две пропорции, равные группы)
p1, p2 = baseline_conversion, variant_conversion
p_bar = (p1 + p2) / 2

sample_size_per_group = ceil(
    ((z_alpha_2 * sqrt(2 * p_bar * (1 - p_bar)) + z_beta * sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2)
    / ((p2 - p1) ** 2)
)

# 4) Длительность эксперимента
users_per_day_per_group = daily_users * traffic_split["A"]
experiment_days = ceil(sample_size_per_group / users_per_day_per_group)

# 5) Метрики
primary_metric = "Conversion Rate (доля пользователей с целевым действием)"
secondary_metrics = [
    "Accuracy модели",
    "Latency инференса (p95)",
    "Доля ошибок сервиса",
]
guards = [
    "p95 latency не хуже контроля более чем на 10%",
    "error rate не выше контроля более чем на 1 п.п.",
]

# 6) Результат плана
ab_test_plan = {
    "hypothesis": hypothesis,
    "alpha": alpha,
    "power": power,
    "baseline_conversion": baseline_conversion,
    "target_conversion_B": variant_conversion,
    "mde_abs": mde_abs,
    "traffic_split": traffic_split,
    "sample_size_per_group": sample_size_per_group,
    "estimated_duration_days": experiment_days,
    "primary_metric": primary_metric,
    "secondary_metrics": secondary_metrics,
    "guardrail_metrics": guards,
    "decision_rule": "Если p-value < alpha и guardrail-метрики не ухудшились, выкатываем B в прод.",
}

print("A/B test plan:")
for k, v in ab_test_plan.items():
    print(f"- {k}: {v}")

A/B test plan:
- hypothesis: Новая модель (B) повысит конверсию в целевое действие по сравнению с текущей моделью (A).
- alpha: 0.05
- power: 0.8
- baseline_conversion: 0.1
- target_conversion_B: 0.12000000000000001
- mde_abs: 0.02
- traffic_split: {'A': 0.5, 'B': 0.5}
- sample_size_per_group: 3837
- estimated_duration_days: 8
- primary_metric: Conversion Rate (доля пользователей с целевым действием)
- secondary_metrics: ['Accuracy модели', 'Latency инференса (p95)', 'Доля ошибок сервиса']
- guardrail_metrics: ['p95 latency не хуже контроля более чем на 10%', 'error rate не выше контроля более чем на 1 п.п.']
- decision_rule: Если p-value < alpha и guardrail-метрики не ухудшились, выкатываем B в прод.


## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

In [47]:
# %%writefile ml_pipeline.py
# import numpy as np
# import pandas as pd
# from sklearn.datasets import load_iris
# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score
# iris = load_iris();X = iris.data ;y = iris.target
# hyperparameters={"n_estimators":100, "random_state":42}
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# model = RandomForestClassifier(**hyperparameters)
# model.fit(X_train, y_train);y_pred = model.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)
# print(f'Точность аccuracy: {accuracy:.2f}')

Проверяем работоспособность пайплайна

In [48]:
# !python ml_pipeline.py

Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

In [54]:
%%writefile .github/workflows/ci.yml
name: CI
on: [push, pull_request]

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v2
    - name: Set up Python
      uses: actions/setup-python@v2
      with:
        python-version: '3.12.12'
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install scikit-learn numpy pandas
    - name: Run pipeline
      run: |
        python -c "import numpy as np; import pandas as pd; from sklearn.datasets import load_iris; from sklearn.model_selection import train_test_split; from sklearn.ensemble import RandomForestClassifier; from sklearn.metrics import accuracy_score; iris = load_iris(); X = iris.data; y = iris.target; X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42); model = RandomForestClassifier(n_estimators=100, random_state=42); model.fit(X_train, y_train); y_pred = model.predict(X_test); accuracy = accuracy_score(y_test, y_pred); print(f'Accuracy: {accuracy:.2f}')"
    - name: Make pipeline reproducible
      run: |
        python -c "print('какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн?')"


Writing .github/workflows/ci.yml


Начинаем отправку в репозиторий

In [50]:
# !git add ./.github/workflows/ci.yml ml_pipeline.py
# !git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitHub Actions"
# !git log

После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

[CI](https://github.com/Lilya-te/ml-ops-hw07-git/actions/workflows/ci.yml)

In [51]:
!git add --all
!git commit -m "final commit"
!git log

[main 3c44b8d] final commit
 1 file changed, 36 insertions(+), 42 deletions(-)
commit 3c44b8d94a963f774002c5da82d7862fe84bb638 (HEAD -> main)
Author: Liliia Tereshchenkova <k.9602585349@gmail.com>
Date:   Sun May 10 17:33:48 2026 +0300

    final commit

commit f1ff842c14ece69deb9283e6f3184b69141a9266 (origin/main)
Author: Liliia Tereshchenkova <k.9602585349@gmail.com>
Date:   Sun May 10 17:33:40 2026 +0300

    first commit

commit 2d74b269fba82b2b64ff9ad6fa66b798e739c901
Author: Liliia Tereshchenkova <k.9602585349@gmail.com>
Date:   Sun May 10 17:32:55 2026 +0300

    final commit

commit e14bc3680fc0cf8c0762bd792f9089ca96f5de62
Author: Liliia Tereshchenkova <k.9602585349@gmail.com>
Date:   Sun May 10 17:32:48 2026 +0300

    first commit


## 6. Итоговое оформление

В итоговых выводах дайте 5–8 предложений о своем опыте работы с инструментами модуля: что оказалось простым, что вызвало трудности, какие выводы сделали по обоснованию стратегии деплоя.

